In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier  
from sklearn.ensemble import RandomForestClassifier  
from sklearn.pipeline import make_pipeline


df=pd.read_csv('gender_classification_v7.csv')
x=df.iloc[:,:-1]
df['gender']=np.where(df['gender']=='Male',1,0)
y=df.iloc[:,-1]
def train_test_validation(x,y,test_size:float=0.25,random_state:int=12,stratify=y):
    from sklearn.model_selection import train_test_split;x_,x_test,y_,y_test=train_test_split(x,y,test_size=test_size,random_state=random_state);x_train,x_val,y_train,y_val=train_test_split(x_,y_,test_size=(test_size/(1-test_size)),random_state=random_state);return x_train,x_test,x_val,y_train,y_test,y_val
x_train,x_test,x_val,y_train,y_test,y_val=train_test_validation(x,y,random_state=12)

In [16]:
pipe1=make_pipeline(StandardScaler(),KNeighborsClassifier(n_jobs=-1,n_neighbors=3))
pipe2=make_pipeline(StandardScaler(),LogisticRegression(random_state=12))
pipe3=make_pipeline(StandardScaler(),DecisionTreeClassifier(random_state=12,max_depth=2))
pipe4=make_pipeline(StandardScaler(),RandomForestClassifier(n_jobs=-1,random_state=12))

results=[]
models={
    'KNN':pipe1,
    'Logr':pipe2,
    'DT':pipe3,
    'RFC':pipe4
}
for name,pipe in models.items():
    result=round(pipe.fit(x_train,y_train).score(x_val,y_val)*100,2)
    print(f"{name} model has acc: {result}");results.append(result)
results

KNN model has acc: 97.28
Logr model has acc: 97.36
DT model has acc: 87.68
RFC model has acc: 97.28


[97.28, 97.36, 87.68, 97.28]

In [17]:
from sklearn.model_selection import GridSearchCV
pipes=[pipe1,pipe2,pipe3,pipe4]
params=[
    {
        'kneighborsclassifier__n_neighbors':[1,2,3,4,5,7,10,12]
    },
    {
        'logisticregression__C':[0.01,.1,1,10,100]
    },
    {
        'decisiontreeclassifier__max_depth':[1,2,3,4,5,7,10]
    },
    {
        'randomforestclassifier__n_estimators':[10,50,100],
        'randomforestclassifier__max_depth':[1,2,3,5,10]
    }
]

best_estimates=[]

for pipe,param in zip(pipes,params):
    grid_search=GridSearchCV(
        pipe,
        param,
        n_jobs=-1,
        scoring='accuracy',
        verbose=0,
        cv=5
    )
    grid_search.fit(x_train,y_train)
    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Best score: {grid_search.best_score_}")
    print(f"Validation score: {grid_search.score(x_val,y_val)}")
    best_estimates.append(grid_search.best_estimator_)

Best Parameters: {'kneighborsclassifier__n_neighbors': 5}
Best score: 0.9687999999999999
Validation score: 0.968
Best Parameters: {'logisticregression__C': 0.01}
Best score: 0.9683999999999999
Validation score: 0.9728
Best Parameters: {'decisiontreeclassifier__max_depth': 7}
Best score: 0.9663999999999999
Validation score: 0.9744
Best Parameters: {'randomforestclassifier__max_depth': 3, 'randomforestclassifier__n_estimators': 50}
Best score: 0.9752000000000001
Validation score: 0.9792


In [25]:
# Using best parameters on test set and comparing results.
for model_name,best_model in zip(models.keys() ,best_estimates):
    print(f"{model_name} has score in test set: {round(best_model.score(x_test,y_test)*100,2)}%")

KNN has score in test set: 96.08%
Logr has score in test set: 96.4%
DT has score in test set: 95.68%
RFC has score in test set: 96.8%


In [32]:
print('Winner is RFC with score of 96.8%.')
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.inspection import permutation_importance
print('\n\n',classification_report(y_test,best_estimates[3].predict(x_test)))
print('\n\n',confusion_matrix(y_test,best_estimates[3].predict(x_test)))
print('\n\n',
      pd.Series(permutation_importance(best_estimates[3],x_test,y_test,n_jobs=-1,n_repeats=10,random_state=12).importances_mean,index=x.columns).sort_values(ascending=False)
      )

Winner is RFC with score of 96.8%.


               precision    recall  f1-score   support

           0       0.96      0.97      0.97       613
           1       0.97      0.97      0.97       638

    accuracy                           0.97      1251
   macro avg       0.97      0.97      0.97      1251
weighted avg       0.97      0.97      0.97      1251



 [[595  18]
 [ 22 616]]


 nose_wide                    0.072982
distance_nose_to_lip_long    0.038849
nose_long                    0.038449
forehead_height_cm           0.023581
forehead_width_cm            0.020783
lips_thin                    0.017826
long_hair                   -0.001119
dtype: float64
